### Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from transformers import HubertForSequenceClassification, Wav2Vec2FeatureExtractor, TrainingArguments, Trainer
from datasets import load_dataset

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
dataset = load_dataset("google/speech_commands", "v0.02", streaming=True)
dataset = list(dataset['train'].take(2000))

speakers = list(set(x['speaker_id'] for x in dataset))
speaker2id = {s: i for i,s in enumerate(speakers)}

feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(
    "facebook/hubert-base-ls960"
)

def preprocess(batch):
    audio = batch['audio']['array']
    max_len=16000
    audio = audio[:max_len] if len(audio) > max_len else np.pad(audio, (0, max_len - len(audio)))
    inputs = feature_extractor(audio, sampling_rate=16000)
    batch['input_values'] = inputs['input_values'][0]
    batch['labels'] = speaker2id[batch["speaker_id"]]
    return batch

dataset = dataset.map(preprocess)

c:\Users\raian\source\repos\AI\.env\Lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for google/speech_commands contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/google/speech_commands
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
model = HubertForSequenceClassification.from_pretrained('facebook/hubert-base-ls960', num_labels = 10)

# for param in model.hubert.parameters():
#     param.requires_grad = False

# # unfreeze last layers
# for layer in model.hubert.encoder.layers[-1:]:
#     for param in layer.parameters():
#         param.requires_grad = True

# Only classifier
for param in model.hubert.parameters():
    param.requires_grad = False

frozen = 0
for name, param in model.named_parameters():
    if not param.requires_grad:
        frozen+=param.numel()

total_params = sum(p.numel() for p in model.parameters())
print(total_params)
print("Trainable part: ", (total_params-frozen)/total_params)

Loading weights: 100%|██████████| 211/211 [00:00<?, ?it/s]
HubertForSequenceClassification LOAD REPORT from: facebook/hubert-base-ls960
Key               | Status  | 
------------------+---------+-
projector.weight  | MISSING | 
classifier.bias   | MISSING | 
projector.bias    | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


94571146
Trainable part:  0.002108825032108631


In [ ]:
dataset = dataset.train_test_split(test_size=0.2)
train_ds, test_ds = dataset['train'], dataset['test']

In [ ]:
args = TrainingArguments(
    output_dir='./hubert',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    learning_rate=1e-4,
    save_strategy='epoch',
    eval_strategy='epoch'
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset,
    eval_dataset=test_ds
)

trainer.train()

In [ ]:
from sklearn.metrics import accuracy_score

def compute_metrics(logits, labels):
    preds = np.argmax(logits)
    return accuracy_score(labels, preds)

# import evaluate
# accuracy_metric = evaluate.load('accuracy')

# def compute_metrics(logits, labels):
#     preds = torch.argmax(logits)
#     return accuracy_metric.compute(predictions=preds, references=labels)